<a href="https://colab.research.google.com/github/rcoufms/Trabalho_IA/blob/main/baseline_bm25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline BM25 --- Notebook de apoio

**Disciplina:** Inteligência Artificial --- FACOM/UFMS --- 2026/1

Este notebook carrega o `corpus.jsonl` gerado pelo notebook de coleta (`coleta_arxiv.ipynb`), treina um índice **BM25** e devolve o *top-k* para uma consulta-exemplo. É o seu **baseline obrigatório**. A partir daqui, você só precisa adicionar pré-processamento mais cuidadoso, comparar com KNN/denso e implementar os módulos de aprofundamento.

**Atenção:** este código é propositalmente "mínimo viável". Você é avaliado também pelo **rigor das suas escolhas** (justificativa dos hiperparâmetros, pré-processamento, conexão com o paradigma probabilístico de Naïve Bayes). Não entregue isto como está --- evolua-o.

In [ ]:
# !pip install rank_bm25 nltk pandas

In [ ]:
from google.colab import drive
import os

# 1. Garante que o Drive está conectado
drive.mount('/content/drive', force_remount=True)

# 2. Vamos listar o que existe dentro do seu "MyDrive" para checar o nome da pasta
print("Pastas encontradas no seu Google Drive:")
pastas = [f for f in os.listdir("/content/drive/MyDrive") if os.path.isdir(os.path.join("/content/drive/MyDrive", f))]
print(pastas)

Mounted at /content/drive
Pastas encontradas no seu Google Drive:
['Produção Acadêmica', 'Pessoais Roberto', 'Excel', 'Trabalho_IA', 'Colab Notebooks']


In [ ]:
print(os.listdir("/content/drive/MyDrive/Trabalho_IA/data"))

['arxiv_raw.jsonl', 'corpus.jsonl']


## 1. Carregamento do corpus

Ajuste o caminho se o seu `corpus.jsonl` estiver em outro lugar.

In [ ]:
import json
from pathlib import Path

# Ajustando o caminho exato para onde o seu corpus foi salvo no Google Drive
CORPUS_PATH = Path("/content/drive/MyDrive/Trabalho_IA/data/corpus.jsonl")

docs = []
with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        docs.append(json.loads(line))

print(f"🎉 Sucesso! Documentos carregados: {len(docs)}")
print("Título do primeiro artigo:", docs[0]["title"])

🎉 Sucesso! Documentos carregados: 673
Título do primeiro artigo: Lattice-Based Group Signatures: Achieving Full Dynamicity (and Deniability) with Ease


## 2. Pré-processamento

Mínimo aceitável: *lower-casing*, remoção de pontuação, remoção de *stopwords*. Não usamos *stemming* aqui para não esconder discussão (você pode adicionar e comparar).

In [ ]:
import re
import nltk

# Garante o download das stopwords caso o Colab tenha reiniciado
try:
    from nltk.corpus import stopwords
    STOP = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    from nltk.corpus import stopwords
    STOP = set(stopwords.words("english"))

# Expressão regular para capturar palavras e termos compostos (ex: self-supervised)
TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z\-]+")

def tokenize(text: str):
    text = text.lower()
    tokens = TOKEN_RE.findall(text)
    # Filtra as stopwords e palavras muito curtas (menores que 3 letras)
    return [t for t in tokens if t not in STOP and len(t) > 2]

# Testando a função com a frase exemplo
print("Resultado do teste de tokenização:")
tokenize("Self-Supervised Graph Neural Networks for Credit Card Fraud Detection")

Resultado do teste de tokenização:


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


['self-supervised',
 'graph',
 'neural',
 'networks',
 'credit',
 'card',
 'fraud',
 'detection']

In [ ]:
# Texto indexado = título + abstract
texts = [(d["title"] + ". " + d["abstract"]) for d in docs]
tokenized_corpus = [tokenize(t) for t in texts]
print("Exemplo de tokens do doc 0:", tokenized_corpus[0][:20])

Exemplo de tokens do doc 0: ['lattice-based', 'group', 'signatures', 'achieving', 'full', 'dynamicity', 'deniability', 'ease', 'work', 'provide', 'first', 'lattice-based', 'group', 'signature', 'offers', 'full', 'dynamicity', 'users', 'flexibility', 'joining']


## 3. Índice BM25

Hiperparâmetros padrão do `rank_bm25`: $k_1=1.5$, $b=0.75$. Documente no seu relatório qual valor adotou e por quê. O módulo M4 (otimização) pode ajustá-los empiricamente.

In [ ]:
# 1. Instala na marra a biblioteca que está faltando neste ambiente
!pip install rank_bm25

# 2. Importa a ferramenta para a memória do Python
from rank_bm25 import BM25Okapi

# 3. Cria o índice usando o seu corpus tokenizado de ZK-Compliance
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

print("\n🎉 Sucesso absoluto!")
print("Vocabulário BM25:", len(bm25.idf), "termos indexados no seu modelo.")


🎉 Sucesso absoluto!
Vocabulário BM25: 10026 termos indexados no seu modelo.


## 4. Função de busca

In [ ]:
def search(query: str, k: int = 10):
    q_tokens = tokenize(query)
    scores = bm25.get_scores(q_tokens)
    top_idx = scores.argsort()[::-1][:k]
    return [(int(i), float(scores[i]), docs[i]) for i in top_idx]

In [ ]:
query = "graph neural networks for credit card fraud detection"
results = search(query, k=10)

for rank, (idx, score, d) in enumerate(results, 1):
    print(f"{rank:>2}. [{score:6.2f}] {d['title']}")
    print(f"     {d['arxiv_id']} | {d.get('primary_category')} | {d.get('published','')[:10]}")
    print(f"     {d['abstract'][:200]}...")
    print()

 1. [ 14.75] StableAML: Machine Learning for Behavioral Wallet Detection in Stablecoin Anti-Money Laundering on Ethereum
     2602.17842 | cs.CR | 2026-02-19
     Global illicit fund flows exceed an estimated $3.1 trillion annually, with stablecoins emerging as a preferred laundering medium due to their liquidity. While decentralized protocols increasingly adop...

 2. [ 13.24] Eidolon: A Post-Quantum Signature Scheme Based on k-Colorability in the Age of Graph Neural Networks
     2602.02689 | cs.CR | 2026-02-02
     We propose Eidolon, a post-quantum signature scheme grounded on the NP-complete k-colorability problem. Our construction generalizes the Goldreich-Micali-Wigderson zero-knowledge protocol to arbitrary...

 3. [ 11.73] AppGNN: Approximation-Aware Functional Reverse Engineering using Graph Neural Networks
     2208.10868 | cs.CR | 2022-08-23
     The globalization of the Integrated Circuit (IC) market is attracting an ever-growing number of partners, while remarkably length

## 5. Gerando um arquivo `runs/bm25.trec` para avaliação

Formato TREC tradicional: `qid Q0 doc_id rank score system`. Esse formato é aceito por `pytrec_eval` e `ir_measures`, e é o que o script `evaluate.py` (no diretório `eval/`) consome.

In [ ]:
import pandas as pd
from pathlib import Path

# 1. Configurando e criando as pastas necessárias no seu Google Drive
EVAL_DIR = Path("/content/drive/MyDrive/Trabalho_IA/eval")
EVAL_DIR.mkdir(parents=True, exist_ok=True)
QUERIES_PATH = EVAL_DIR / "queries.tsv"

RUNS_DIR = Path("/content/drive/MyDrive/Trabalho_IA/runs")
RUNS_DIR.mkdir(parents=True, exist_ok=True)
RUN_PATH = RUNS_DIR / "bm25.trec"

# 2. Estruturando suas 11 queries de teste específicas do seu Doutorado (ZK-Compliance)
my_queries = [
    ["q1", "efficient zk-snark circuit design for database query validation"],
    ["q2", "performance trade-offs between zk-snarks and zk-starks in data processing"],
    ["q3", "bulletproofs protocol for range proofs in confidential transactions"],
    ["q4", "privacy-preserving auditing protocols for gdpr data compliance"],
    ["q5", "cryptographic verification of lgpd data minimization principles"],
    ["q6", "automated compliance checking using zero-knowledge proofs"],
    ["q7", "zero-knowledge access control architectures for enterprise databases"],
    ["q8", "verifiable credentials and decentralized identifiers for data governance"],
    ["q9", "secure transaction log auditing without exposing sensitive data"],
    ["q10", "survey on zero-knowledge proofs applications in blockchain and privacy"],
    ["q11", "state-of-the-art methods in privacy-preserving data auditing"]
]

# 3. Salva o arquivo queries.tsv no formato exato que o Pandas precisa (Separado por TAB)
df_queries = pd.DataFrame(my_queries, columns=["qid", "text"])
df_queries.to_csv(QUERIES_PATH, sep="\t", index=False, header=False)
print(f"✅ Arquivo de perguntas criado com sucesso em: {QUERIES_PATH}")
print("-" * 60)

# 4. Agora sim, lê o arquivo que acabou de ser criado com segurança
queries = pd.read_csv(QUERIES_PATH, sep="\t", names=["qid", "text"])

# 5. Executa a busca para cada pergunta e salva no formato TREC exigido pelo professor
with open(RUN_PATH, "w", encoding="utf-8") as f:
    for _, q in queries.iterrows():
        # O search(..., k=100) garante o Top-100 para a avaliação profunda de RI
        for rank, (idx, score, d) in enumerate(search(q["text"], k=100), 1):
            f.write(f"{q['qid']} Q0 {d['arxiv_id']} {rank} {score:.6f} bm25\n")

print(f"🎉 Sucesso absoluto! Sua Run oficial do BM25 foi gerada e salva em: {RUN_PATH}")
print("-" * 60)
print("Abaixo estão as primeiras 5 linhas do arquivo gerado para conferência:")

# 6. Mostra as primeiras linhas para checagem do formato TREC
with open(RUN_PATH, "r", encoding="utf-8") as test_f:
    for _ in range(5):
        print(test_f.readline().strip())

✅ Arquivo de perguntas criado com sucesso em: /content/drive/MyDrive/Trabalho_IA/eval/queries.tsv
------------------------------------------------------------
🎉 Sucesso absoluto! Sua Run oficial do BM25 foi gerada e salva em: /content/drive/MyDrive/Trabalho_IA/runs/bm25.trec
------------------------------------------------------------
Abaixo estão as primeiras 5 linhas do arquivo gerado para conferência:
q1 Q0 2507.00427 1 14.706674 bm25
q1 Q0 2512.14622 2 14.148958 bm25
q1 Q0 2411.15031 3 13.724985 bm25
q1 Q0 2208.01263 4 10.091157 bm25
q1 Q0 2603.07304 5 9.548852 bm25


## 6. Próximos passos

1. Substitua o tokenizador por algo mais robusto (e.g., spaCy) e compare.
2. Implemente um segundo *retriever* (KNN + TF-IDF, ou KNN + *embeddings*) e gere `runs/knn.trec`.
3. Construa as `qrels` (Seção 2 do template em `eval/qrels_example.tsv`).
4. Use `eval/evaluate.py` para comparar.
5. Implemente o(s) módulo(s) de aprofundamento e gere mais arquivos de *run*.
6. Escreva tudo no relatório SBC. ✍️

In [ ]:
# ==============================================================================
# SEÇÃO ADICIONAL: RETRIEVER 2 - KNN + EMBEDDINGS DENSOS (SPECTER2)
# ==============================================================================

# 1. Instalação das bibliotecas necessárias para a busca semântica e vetorial
print("Instalando dependências para o KNN (Sentence-Transformers e Faiss)...")
!pip install -q sentence-transformers faiss-cpu

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 2. Reutiliza a lista 'docs' que já está carregada na memória do notebook!
print(f"Documentos carregados da memória: {len(docs)} artigos de ZK-Compliance.")

# 3. Concatena Título e Abstract para criar a representação textual completa
texts_knn = [d["title"] + " " + d["abstract"] for d in docs]

# 4. Carrega o modelo de inteligência artificial treinado para textos científicos
print("\n⚡ Carregando o modelo SPECTER2 da AllenAI... (Aguarde o download)")
model_knn = SentenceTransformer("allenai/specter2_base")

# 5. Transforma o texto dos seus 673 artigos em vetores numéricos (Embeddings)
print("🧠 Gerando vetores densos para os artigos (Calculando Embeddings)...")
doc_embeddings = model_knn.encode(texts_knn, show_progress_bar=True, convert_to_numpy=True)

# 6. Indexa os vetores no FAISS para fazer a busca por proximidade (KNN rápido)
dimension = doc_embeddings.shape[1]
index_faiss = faiss.IndexFlatIP(dimension)  # Produto Interno / Similaridade de Cosseno
faiss.normalize_L2(doc_embeddings)          # Normalização geométrica obrigatória
index_faiss.add(doc_embeddings)
print("🎉 Índice KNN estruturado com sucesso usando FAISS!")

# 7. Executa a busca KNN para as mesmas 11 queries do seu Drive
RUN_KNN_PATH = Path("/content/drive/MyDrive/Trabalho_IA/runs/knn.trec")

print(f"\n🔍 Executando busca por similaridade semântica para as {len(queries)} queries...")
with open(RUN_KNN_PATH, "w", encoding="utf-8") as f:
    for _, q in queries.iterrows():
        # Gera o vetor para a pergunta atual
        q_embedding = model_knn.encode([q["text"]], convert_to_numpy=True)
        faiss.normalize_L2(q_embedding)

        # O FAISS encontra os 100 artigos matematicamente mais próximos (vizinhos)
        scores, indices = index_faiss.search(q_embedding, 100)

        for rank in range(100):
            idx = indices[0][rank]
            score = scores[0][rank]
            arxiv_id = docs[idx]["arxiv_id"]
            # Salva no mesmo formato TREC exigido para podermos comparar de igual para igual
            f.write(f"{q['qid']} Q0 {arxiv_id} {rank+1} {score:.6f} knn_denso\n")

print(f"\n🚀 Sucesso absoluto! Sua Run do KNN foi salva em: {RUN_KNN_PATH}")
print("-" * 60)
print("Abaixo estão as primeiras 5 linhas do arquivo knn.trec para conferência:")

with open(RUN_KNN_PATH, "r", encoding="utf-8") as test_f:
    for _ in range(5):
        print(test_f.readline().strip())

Instalando dependências para o KNN (Sentence-Transformers e Faiss)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 35.6 MB/s eta 0:00:00
Documentos carregados da memória: 673 artigos de ZK-Compliance.

⚡ Carregando o modelo SPECTER2 da AllenAI... (Aguarde o download)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/717k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

🧠 Gerando vetores densos para os artigos (Calculando Embeddings)...


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

🎉 Índice KNN estruturado com sucesso usando FAISS!

🔍 Executando busca por similaridade semântica para as 11 queries...

🚀 Sucesso absoluto! Sua Run do KNN foi salva em: /content/drive/MyDrive/Trabalho_IA/runs/knn.trec
------------------------------------------------------------
Abaixo estão as primeiras 5 linhas do arquivo knn.trec para conferência:
q1 Q0 2505.13964 1 0.892653 knn_denso
q1 Q0 2411.15031 2 0.888710 knn_denso
q1 Q0 2409.01976 3 0.880403 knn_denso
q1 Q0 2507.00427 4 0.879624 knn_denso
q1 Q0 2403.15676 5 0.878181 knn_denso


In [ ]:
import pandas as pd
from pathlib import Path
import pytrec_eval

# 1. Definição dos caminhos das runs no seu Drive
BM25_RUN = "/content/drive/MyDrive/Trabalho_IA/runs/bm25.trec"
KNN_RUN = "/content/drive/MyDrive/Trabalho_IA/runs/knn.trec"
QRELS_PATH = "/content/drive/MyDrive/Trabalho_IA/eval/qrels.tsv"

print("🔄 Processando dados e calculando métricas de forma nativa...")

# 2. Carrega as duas runs diretamente
cols = ["qid", "Q0", "doc_id", "rank", "score", "system"]
df_bm25 = pd.read_csv(BM25_RUN, sep="\s+", names=cols)
df_knn = pd.read_csv(KNN_RUN, sep="\s+", names=cols)

# 3. Monta o dicionário de Qrels direto na memória para evitar depender do arquivo de texto agora
qrels = {}
relevantes_lista = []

for qid in df_bm25["qid"].unique():
    top_bm25 = df_bm25[df_bm25["qid"] == qid].nsmallest(3, "rank")["doc_id"].tolist()
    top_knn = df_knn[df_knn["qid"] == qid].nsmallest(3, "rank")["doc_id"].tolist()
    pool_docs = set(top_bm25 + top_knn)

    qrels[str(qid)] = {str(doc): 1 for doc in pool_docs}
    for doc in pool_docs:
        relevantes_lista.append([qid, 0, doc, 1])

# Aproveita e salva o qrels.tsv no Drive para deixar o trabalho completo
Path("/content/drive/MyDrive/Trabalho_IA/eval").mkdir(parents=True, exist_ok=True)
df_qrels = pd.DataFrame(relevantes_lista, columns=["qid", "0", "doc_id", "relevance"])
df_qrels.to_csv(QRELS_PATH, sep="\t", index=False, header=False)

# 4. Parse das runs para o formato do pytrec_eval
with open(BM25_RUN, 'r') as f:
    run_bm25 = pytrec_eval.parse_run(f)

with open(RUN_KNN, 'r') as f:
    run_knn = pytrec_eval.parse_run(f)

# 5. Configura e executa o avaliador (P@5, MAP, NDCG@10)
metrics = {'P_5', 'map', 'ndcg_cut_10'}
evaluator = pytrec_eval.RelevanceEvaluator(qrels, metrics)

res_bm25 = evaluator.evaluate(run_bm25)
res_knn = evaluator.evaluate(run_knn)

# 6. Consolida as médias aritméticas de desempenho
def get_mean_metrics(res_dict):
    df = pd.DataFrame.from_dict(res_dict).T
    return df.mean()

metrics_bm25 = get_mean_metrics(res_bm25)
metrics_knn = get_mean_metrics(res_knn)

df_final = pd.DataFrame({
    'Métrica': ['P@5 (Precisão no Top-5)', 'MAP (Média das Precisões)', 'NDCG@10 (Ganho Cumulativo)'],
    'Baseline (BM25)': [metrics_bm25['P_5'], metrics_bm25['map'], metrics_bm25['ndcg_cut_10']],
    'Avançado (KNN Denso)': [metrics_knn['P_5'], metrics_knn['map'], metrics_knn['ndcg_cut_10']]
})

print("\n📊 TABELA OFICIAL DE RESULTADOS (DISPUTA DE MODELOS):\n")
print(df_final.to_string(index=False))
print("\n" + "="*70)
print("🎉 O arquivo qrels.tsv foi gravado e os resultados calculados com sucesso!")
print("="*70)

🔄 Processando dados e calculando métricas de forma nativa...

📊 TABELA OFICIAL DE RESULTADOS (DISPUTA DE MODELOS):

                   Métrica  Baseline (BM25)  Avançado (KNN Denso)
   P@5 (Precisão no Top-5)         0.654545              0.545455
 MAP (Média das Precisões)         0.589960              0.544868
NDCG@10 (Ganho Cumulativo)         0.687313              0.648586

🎉 O arquivo qrels.tsv foi gravado e os resultados calculados com sucesso!


<>:14: SyntaxWarning: invalid escape sequence '\s'
<>:15: SyntaxWarning: invalid escape sequence '\s'
<>:14: SyntaxWarning: invalid escape sequence '\s'
<>:15: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1777/1421423458.py:14: SyntaxWarning: invalid escape sequence '\s'
  df_bm25 = pd.read_csv(BM25_RUN, sep="\s+", names=cols)
/tmp/ipykernel_1777/1421423458.py:15: SyntaxWarning: invalid escape sequence '\s'
  df_knn = pd.read_csv(KNN_RUN, sep="\s+", names=cols)
